# Continuous Cycle VAD — IPAD / Google Colab

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1. 내장 구현 복원 + 의존성 설치 (Colab의 CUDA PyTorch는 유지)
import base64, hashlib, io, json, os, pathlib, subprocess, sys, zipfile
PROJECT = pathlib.Path('/content/cycle-vad')
PAYLOAD = "..."
payload = base64.b64decode(PAYLOAD)
assert hashlib.sha256(payload).hexdigest() == "51954b3dc34a6f03529a5e6597a85967ceea48454e35e0f069c57efba1490159"
PROJECT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    for member in archive.infolist():
        target = (PROJECT / member.filename).resolve()
        if PROJECT.resolve() not in target.parents:
            raise ValueError('Invalid embedded member')
    archive.extractall(PROJECT)
os.environ['PYTHONPATH'] = str(PROJECT / 'src')
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['OPENBLAS_NUM_THREADS'] = '2'
sys.path.insert(0, str(PROJECT / 'src'))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT / 'requirements-cycle-colab.txt')], check=True)
print('Implementation restored:', PROJECT)


In [ ]:
# 2. GPU 확인 + Google Drive 연결
import torch
from google.colab import drive
print('PyTorch:', torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('런타임 유형을 GPU로 변경한 후 다시 실행하세요.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
drive.mount('/content/drive')


In [ ]:
# 3. 이 셀의 경로를 수정하세요.
from pathlib import Path
DATA_ROOT = Path('/content/drive/MyDrive/IPAD_dataset')
OUTPUT_DIR = Path('/content/drive/MyDrive/IVAD_cycle_runs/cycle_dinov2_base_v1')
SCENES = ['R02']  # 첫 실행은 ['R01']로 줄여도 됩니다.
STAGE_TO_LOCAL = False  # 작은 JPG를 많이 읽으므로 여유 디스크가 있으면 True 권장
BOUNDARIES_JSON = None  # 예: '/content/drive/MyDrive/cycle_boundaries.json'
GROUPS_JSON = None      # 예: '/content/drive/MyDrive/recording_groups.json'
STAGE = 'all'           # all / extract / fit / evaluate

config = json.loads((PROJECT / 'configs/cycle_colab.json').read_text())
config['scenes'] = SCENES
# 기본: DINOv2-base 336px, stride=2, FP16 batch=8, OOM 자동 축소
# 최종 짧은 이상 평가에서 stride=1을 별도 검증할 수 있습니다(추출량 약 2배).
# config['stride'] = 1
# 더 작은 GPU: model_id='facebook/dinov2-small', image_size=224, batch_size=4
# CLIP 대조: family='clip', model_id='openai/clip-vit-base-patch16', image_size=224

assert DATA_ROOT.is_dir(), f'폴더를 찾을 수 없습니다: {DATA_ROOT}'
for scene in SCENES:
    assert (DATA_ROOT / scene / 'training' / 'frames').is_dir(), scene
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = PROJECT / 'configs/runtime.json'
CONFIG_PATH.write_text(json.dumps(config, indent=2))
print('Output:', OUTPUT_DIR)
print('Cycle labels:', BOUNDARIES_JSON or 'weak_recording_alignment (실제 cycle GT 미확인)')


## 선택: Drive 폴더를 로컬 디스크로 복사

`STAGE_TO_LOCAL=False`이면 Drive를 직접 읽습니다. True이면 선택한 장면만 `/content/IPAD_cycle_local`로 복사합니다. 첫 복사는 시간이 걸릴 수 있습니다. 기존 파일이 크기·수정시간까지 같으면 건너뜁니다. 원본 Drive 파일은 수정하지 않습니다. 특징 캐시와 모델은 OUTPUT_DIR에 계속 보존됩니다.


In [ ]:
# 4. 선택적 로컬 복사
import shutil
RUN_DATA_ROOT = DATA_ROOT
if STAGE_TO_LOCAL:
    RUN_DATA_ROOT = Path('/content/IPAD_cycle_local')
    needed = sum(p.stat().st_size for scene in SCENES for p in (DATA_ROOT/scene).rglob('*') if p.is_file())
    free = shutil.disk_usage('/content').free
    if needed * 1.1 > free:
        raise RuntimeError(f'로컬 여유 공간 부족: 필요 약 {needed/2**30:.1f} GiB, 여유 {free/2**30:.1f} GiB')
    def copy_changed(src, dst):
        src, dst = Path(src), Path(dst)
        if dst.exists() and src.stat().st_size == dst.stat().st_size and src.stat().st_mtime_ns == dst.stat().st_mtime_ns:
            return str(dst)
        return shutil.copy2(src, dst)
    for scene in SCENES:
        print('Copying:', scene, flush=True)
        shutil.copytree(DATA_ROOT/scene, RUN_DATA_ROOT/scene, dirs_exist_ok=True, copy_function=copy_changed)
print('Input:', RUN_DATA_ROOT)


In [ ]:
path = PROJECT / "tests/test_cycle_vad.py"
text = path.read_text(encoding="utf-8")

args = (
    'fitted.score(synthetic(10))["combined"], '
    'restored.score(synthetic(10))["combined"]'
)
old = f"np.testing.assert_array_equal({args})"
new = f"np.testing.assert_allclose({args}, rtol=1e-6, atol=1e-7)"

assert old in text or new in text, "테스트 파일 내용 확인이 필요합니다."
path.write_text(text.replace(old, new), encoding="utf-8")
print("테스트 비교 조건 수정 완료")

In [ ]:
# 5. 코드 smoke/regression 검증 (IPAD 성능 실험이 아닙니다)
subprocess.run([sys.executable, '-m', 'pytest', str(PROJECT / 'tests/test_cycle_vad.py'), '-q'],
               cwd=PROJECT, check=True)


In [ ]:
# 6. 특징 추출 → 정상 모델 학습/선택 → 보정 → 평가
# 중단되면 동일 OUTPUT_DIR로 다시 실행: 완료한 영상의 캐시를 재사용합니다.
command = [sys.executable, '-m', 'cycle_vad.pipeline',
           '--data-root', str(RUN_DATA_ROOT), '--output', str(OUTPUT_DIR),
           '--config', str(CONFIG_PATH), '--stage', STAGE]
if BOUNDARIES_JSON:
    command += ['--boundaries', str(BOUNDARIES_JSON)]
if GROUPS_JSON:
    command += ['--groups', str(GROUPS_JSON)]
print('Running:', ' '.join(command), flush=True)
subprocess.run(command, cwd=PROJECT, check=True)


In [ ]:
# 7. 장면별 결과와 macro 평균
import pandas as pd
from IPython.display import display
results_path = OUTPUT_DIR / 'metrics_by_scene.json'
if results_path.exists():
    results = json.loads(results_path.read_text())
    display(pd.DataFrame([dict(scene=scene, branch=branch, **metrics)
                          for scene, branches in results.items() for branch, metrics in branches.items()]))
    display(pd.DataFrame(json.loads((OUTPUT_DIR/'macro_metrics.json').read_text())).T)
else:
    print('evaluate 또는 all 단계를 실행하면 평가 결과가 생성됩니다.')


In [ ]:
# 8. 테스트 영상 하나의 점수·cycle 좌표·confidence 확인
import numpy as np
import matplotlib.pyplot as plt
SCENE = SCENES[0]
prediction_files = sorted((OUTPUT_DIR / SCENE / 'predictions').glob('testing_*.npz'))
if prediction_files:
    selected = prediction_files[0]  # 원하는 영상 파일로 변경 가능
    with np.load(selected, allow_pickle=False) as p:
        ids = p['indices']
        fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
        thresholds = json.loads((OUTPUT_DIR / SCENE / 'fit_report.json').read_text())['thresholds']
        for branch in ['appearance', 'cycle_conditioned', 'combined']:
            axes[0].plot(ids, p[branch], label=branch)
        axes[0].axhline(thresholds['combined'], color='red', linestyle='--', label='normal holdout threshold')
        axes[0].legend(); axes[0].set_ylabel('Anomaly score')
        axes[1].plot(ids, p['angle']); axes[1].set_ylabel('Cycle position [0,1)')
        axes[2].plot(ids, p['confidence']); axes[2].set_ylabel('Cycle confidence')
        axes[2].set_xlabel('Source frame index')
        fig.suptitle(selected.name)
        plt.tight_layout(); plt.show()
else:
    print('저장된 테스트 prediction이 없습니다.')


In [ ]:
# 9. 거친 지역 이상 근거 확인 (픽셀 GT localization 결과가 아닙니다)
from PIL import Image, ImageOps
from cycle_vad.data import frame_paths
if prediction_files:
    with np.load(selected, allow_pickle=False) as p:
        sample = int(np.argmax(p['combined']))
        frame_id = int(p['indices'][sample])
        local_distance = p['patch_distance'][sample].reshape(config['encoder']['patch_grid'], -1)
    seq = selected.stem.removeprefix('testing_')
    frame = frame_paths(RUN_DATA_ROOT/SCENE/'testing'/'frames'/seq)[frame_id]
    image = Image.open(frame).convert('RGB')
    size = config['encoder']['image_size']
    canvas = ImageOps.pad(image, (size, size), color=(124, 116, 104))
    heat = Image.fromarray(local_distance).resize((size, size), Image.Resampling.BILINEAR)
    fig, ax = plt.subplots(figsize=(9, 7))
    ax.imshow(canvas); overlay = ax.imshow(np.array(heat), cmap='magma', alpha=.5)
    ax.set_title(f'{SCENE}/{seq} — frame {frame_id} — local feature distance')
    ax.axis('off'); fig.colorbar(overlay, ax=ax); plt.show()
